In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/indra22/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/indra22/tugas4/
print("Berhasil diunggah ke HDFS: /user/indra22/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/indra22/tugas4/transaksi_september_2026.csv


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

spark = SparkSession.builder \
    .appName("Tugas4_Indra") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [6]:
df = spark.read.csv (
    "hdfs://localhost:9000/user/indra22/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

print("Tipe objek: ", type(df))
df.printSchema()

print("Jumlah baris: ", df.count())

print("\nData 10 baris pertama: ")
df.show(10)

Tipe objek:  <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris:  1000

Data 10 baris pertama: 
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wall

In [20]:
jumlah_kosong = df.filter(col("rating").isNull()).count()
print("Jumlah rating kosong:", jumlah_kosong)

# df.na.drop()
df_bersih = df.na.drop(subset=["rating"])

print("Jumlah baris sebelum:", df.count())
print("Jumlah baris setelah drop:", df_bersih.count())
print("Cek ulang rating kosong:", df_bersih.filter(col("rating").isNull()).count())

Jumlah rating kosong: 204
Jumlah baris sebelum: 1000
Jumlah baris setelah drop: 796
Cek ulang rating kosong: 0


In [8]:
df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

df = df.withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar")
    .otherwise("Kecil")
)

df.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi"
).show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [10]:
# Kategori dengan Total Pendapatan Tertinggi

ringkasan_kategori = df.groupBy("kategori") \
    .agg(
        spark_sum("total_pendapatan").alias("total_pendapatan")
    ) \
    .orderBy(
        col("total_pendapatan").desc()
    )

ringkasan_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [12]:
# Kota dengan Transaksi Tier "Besar" Terbanyak

ringkasan_kota = df.filter(
    col("tier_transaksi") == "Besar"
).groupBy("kota") \
    .count() \
    .orderBy(
        col("count").desc()
    )

ringkasan_kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [21]:
# Rata-rata Rating Berdasarkan Metode Pembayaran

rata_rating = df.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(col("rata_rata_rating").desc())

rata_rating.show()

+-----------------+-----------------+
|metode_pembayaran| rata_rata_rating|
+-----------------+-----------------+
|              COD|4.172413793103448|
|    Transfer Bank| 4.16256157635468|
|         E-Wallet|4.135678391959799|
|     Kartu Kredit|4.109947643979058|
+-----------------+-----------------+



In [16]:
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("hdfs://localhost:9000/user/indra22/tugas4/hasil_data_transaksi")

print("Data berhasil disimpan ke HDFS.")
!hdfs dfs -ls /user/indra22/tugas4/hasil_data_transaksi

Data berhasil disimpan ke HDFS.
Found 2 items
-rw-r--r--   3 indra22 supergroup          0 2026-09-16 20:19 /user/indra22/tugas4/hasil_data_transaksi/_SUCCESS
-rw-r--r--   3 indra22 supergroup      96684 2026-09-16 20:19 /user/indra22/tugas4/hasil_data_transaksi/part-00000-8412b145-95a8-4734-b8cb-5a685ef729f7-c000.csv
